# How learning actually happens

> Gradient descent from scratch: the loop that trains everything from a two-parameter line to a frontier language model, and the three ways it goes wrong.

Read this chapter at `/learn/05-how-learning-happens/`. Exported from `src/content/chapters/05-how-learning-happens.mdx` — edit there, not here.


Yesterday's closed-form solution works for linear least squares. And for nothing
else, ever.

Today you learn the method that works for *everything* — and here's the thing I'd
love you to hold onto as we go: at the scale of a frontier language model, with
its four hundred billion parameters and its warehouse of GPUs, the training loop
is still recognisably the fifteen lines you're about to write.

Not analogous to them. Recognisably them.

## The idea

You're standing on a hillside in thick fog. You can't see the valley floor. You
have no map.

But you can feel the slope under your feet. So you take a small step downhill.
And then you do it again.

That's it. That is gradient descent, and there is
genuinely nothing more to it. The gradient tells you which
way is uphill; you go the other way; the step size is a number you choose and
then immediately regret.

Every optimiser you will ever meet — SGD, momentum, RMSProp, Adam, AdamW, Lion,
Shampoo — is this loop with a different rule for computing the step.

Not a different algorithm. A different *step rule*. When you read a paper
introducing a new optimiser, that's the only thing you're looking for.

## Let's derive the gradient once, by hand

We need $\partial L/\partial w$ and $\partial L/\partial b$ for
$L = \frac{1}{n}\sum (wx_i + b - y_i)^2$. It's two lines of calculus and I'd
rather do it with you than assert it.

Write the residual for one example as $r_i = \hat{y}_i - y_i = (wx_i + b) - y_i$,
so that $L = \frac{1}{n}\sum r_i^2$. All we've done is give the miss a name.

Now apply the chain rule. The outer function is
$r \mapsto r^2$, whose derivative is $2r$. The inner function is
$w \mapsto wx_i + b - y_i$, whose derivative with respect to $w$ is $x_i$, and
with respect to $b$ is $1$.

Multiply them together:

$$
\frac{\partial L}{\partial w} = \frac{1}{n}\sum_i 2 r_i \, x_i
\qquad
\frac{\partial L}{\partial b} = \frac{1}{n}\sum_i 2 r_i
$$

Done. But please don't stop there, because the formulas are much less interesting
than what they *say*.

**Read the first one in words:** the gradient with respect to $w$ is *the average
miss, weighted by the input that produced it.*

Think about why that's exactly right. An example way out at $x = 10$ has more
influence on the slope than one at $x = 0.1$ — and it should, because tilting the
line moves distant points a great deal and nearby points hardly at all. The
calculus knew that. We didn't have to tell it.

**Read the second one:** the gradient with respect to $b$ is *just the average
miss.*

If you're predicting too high on average, lower the intercept. Which is so
obvious you could have guessed it over breakfast — and there is something quietly
reassuring about the calculus agreeing with you.

That factor of 2 is real, and essentially everybody drops it.

Some textbooks define MSE with a $\frac{1}{2n}$ out front purely so the 2
cancels and the formulas look tidier. It makes no difference to *where* the
minimum is — it rescales every gradient by the same constant, which is
indistinguishable from halving the learning rate. So people quietly absorb it and
move on. Now you know why the formula sometimes has a 2 and sometimes doesn't.

Right. Fifteen lines, no library:

In [ ]:
import numpy as np, matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n = 60
hours = rng.uniform(0, 10, n)
score = 12 + 7.5 * hours + rng.normal(0, 8, n)

def fit(x, y, lr=0.01, steps=200):
    w, b = 0.0, 0.0                      # start anywhere
    history = []
    for _ in range(steps):
        pred = w * x + b
        resid = pred - y
        grad_w = 2 * (resid * x).mean()  # dL/dw
        grad_b = 2 * resid.mean()        # dL/db
        w -= lr * grad_w                 # step downhill
        b -= lr * grad_b
        history.append((w, b, (resid ** 2).mean()))
    return w, b, np.array(history)

w, b, hist = fit(hours, score)
print(f"learned  w={w:.3f}  b={b:.3f}   (truth 7.5, 12.0)")
print(f"final loss {hist[-1, 2]:.2f}")

That loop is the whole of training. I'd like to say that again, because it's easy
to nod past: **that loop is the whole of training.** Everything in the remaining
eleven chapters is refinement, scaling, and better ways to compute those two
gradient lines.

Now look at the output. `w` has essentially arrived at 7.5. `b` hasn't — it's
nowhere near 12 after 200 steps.

That's not a bug. That's the next section, and it's more interesting than the
success.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.5, 3))
ax[0].plot(hist[:, 2]); ax[0].set_xlabel("step"); ax[0].set_ylabel("loss")
ax[0].set_title("loss curve"); ax[0].set_yscale("log")
ax[1].plot(hist[:, 0], hist[:, 1], lw=1); ax[1].scatter(7.5, 12, c="crimson", marker="*", s=90)
ax[1].set_xlabel("w"); ax[1].set_ylabel("b"); ax[1].set_title("path through parameter space")
plt.tight_layout()

The right-hand panel is your walk down yesterday's contour plot, drawn as an
actual path. There you are, in the fog, feeling your way.

And look at its shape: it *sprints* along `w` and then crawls along `b`. The
valley is far steeper in one direction than the other, so the same step size is
simultaneously too big for one axis and far too small for the other.

Hold onto that observation. It's the entire reason momentum and Adam had to be
invented, and we'll come back to it in twenty minutes.

## The learning rate is the whole ballgame

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.8), sharey=True)
for ax, lr in zip(axes, [0.001, 0.02, 0.045]):
    _, _, h = fit(hours, score, lr=lr, steps=120)
    ax.plot(np.clip(h[:, 2], 1e-2, 1e7))
    ax.set_yscale("log"); ax.set_title(f"lr = {lr}"); ax.set_xlabel("step")
axes[0].set_ylabel("loss (log)")
plt.tight_layout()

Three pictures, three stories.

Too small and you crawl. The shape is right, the direction is right, and you will
simply run out of patience before you run out of hill.

Too large and something more dramatic happens: you overshoot the valley entirely,
land *higher up* the opposite wall, overshoot harder on the way back, and reach
infinity in about twenty steps. The fog metaphor holds — you took a stride so
long you crossed the whole valley and ended up further up than you started.

**A loss of `nan` almost always means the learning rate is too high.**

Check this first — before the data, before the model, before the code. Here's the
sequence, so you can recognise it: loss grows, weights grow, an activation
overflows to `inf`, then `inf - inf` produces `nan`, and from that moment `nan`
contaminates every parameter on every subsequent update and never, ever leaves.

Divide the learning rate by ten. Try again. You will do this many times, and it
stops feeling like a failure quite quickly.

The practical procedure is to try a few on a log scale — `1e-1, 1e-2, 1e-3,
1e-4` — and take the largest one that doesn't explode.

There's a more systematic version called the **learning-rate finder**, which
fastai popularised: sweep the rate upward over a single short run and plot loss
against rate.

In [ ]:
rates, final = [], []
for lr in np.logspace(-4, -1.2, 40):
    _, _, h = fit(hours, score, lr=lr, steps=60)
    rates.append(lr); final.append(min(h[-1, 2], 1e6))

plt.figure(figsize=(5, 3))
plt.plot(rates, final); plt.xscale("log"); plt.yscale("log")
plt.xlabel("learning rate"); plt.ylabel("loss after 60 steps")
plt.axvline(rates[int(np.argmin(final))], ls=":", c="crimson")
plt.tight_layout()

Pick a rate a little below where the curve turns back upward. That one plot
replaces a great deal of guessing, and it costs you a single short training run.
It's one of the best effort-to-value trades in the whole field.

## Stochastic, mini-batch, and full batch

Our loop uses all 60 examples for every step. With 60 *million* examples, that's
one step per several minutes — and since we just established you need thousands
of steps, that's untenable.

So here's the question: do we actually need all the data to know which way is
downhill?

In [ ]:
def fit_minibatch(x, y, lr=0.01, epochs=40, batch=8, seed=0):
    rng = np.random.default_rng(seed)
    w, b, hist = 0.0, 0.0, []
    for _ in range(epochs):
        order = rng.permutation(len(x))          # reshuffle every epoch
        for s in range(0, len(x), batch):
            idx = order[s:s + batch]
            xb, yb = x[idx], y[idx]
            resid = (w * xb + b) - yb
            w -= lr * 2 * (resid * xb).mean()
            b -= lr * 2 * resid.mean()
        hist.append(((w * x + b - y) ** 2).mean())
    return w, b, np.array(hist)

w2, b2, h2 = fit_minibatch(hours, score)
print(f"mini-batch: w={w2:.3f} b={b2:.3f}  loss={h2[-1]:.2f}")
print(f"full batch: w={w:.3f} b={b:.3f}  loss={hist[-1,2]:.2f}")

No, it turns out. Eight examples will do.

Two words to collect here. An **epoch** is one full pass over the data. A
**batch** is the group of examples used for a single update.

Estimating the gradient from 8 examples instead of 60 gives you a noisier
direction — but it's nearly eight times cheaper, which buys you nearly eight
times as many steps for the same compute. And more noisy steps beat fewer clean
ones, almost every time.

This is one of my favourite things in the subject, so let me take a moment.

You'd assume the noisy gradient is simply the price you pay for going fast — a
regrettable approximation you'd remove if you could afford to.

It isn't. The noise is *doing something for you*.

Picture the fog again, but now the valley floor isn't smooth. It's pocked with
little dimples — narrow local minima where a careful walker would settle down,
declare victory, and stop. A perfectly clean gradient would walk you into the
first dimple you met and hold you there forever, because from inside a dimple,
every direction is uphill.

A noisy gradient stumbles. It takes a step slightly wrong, trips out of the
dimple, and carries on down the real hill.

And there's more. There's reasonable evidence that mini-batch noise biases
training toward *flat* minima rather than sharp ones — broad basins rather than
narrow crevices. Flat minima generalise better, because a flat minimum means the
answer doesn't change much if the world shifts slightly, and the world always
shifts slightly.

So the approximation we made to go faster also made the answer better. That
happens embarrassingly rarely in engineering, and when it does it's worth
noticing.

(A terminology note while we're here: "SGD" — stochastic gradient descent —
technically means *one* example at a time. In practice everybody says SGD and
means mini-batch. Nobody will correct you.)

Batch size is mostly decided by what fits in memory, which is an unglamorous
truth about a much-discussed hyperparameter.

The useful rule of thumb: doubling the batch size lets you increase the learning
rate by roughly $\sqrt{2}$, because your gradient estimate got that much less
noisy and you can afford to trust it further.

## Momentum, and why Adam exists

Remember the parameter-space path: fast along one axis, painfully slow along the
other. Let's fix it.

Momentum's idea is to stop stepping on the raw gradient, and instead accumulate a
**velocity**.

In [ ]:
def fit_general(x, y, lr=0.01, steps=200, beta=0.0):
    w = np.zeros(2)                       # [b, w]
    X = np.column_stack([np.ones(len(x)), x])
    v = np.zeros(2)
    hist = []
    for _ in range(steps):
        g = 2 * X.T @ (X @ w - y) / len(x)
        v = beta * v + (1 - beta) * g     # exponential moving average
        w -= lr * v
        hist.append(((X @ w - y) ** 2).mean())
    return w, np.array(hist)

_, plain    = fit_general(hours, score, beta=0.0)
_, momentum = fit_general(hours, score, beta=0.9)

plt.figure(figsize=(5.2, 3))
plt.plot(plain, label="SGD"); plt.plot(momentum, label="momentum 0.9")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend()
plt.tight_layout()

The intuition is physical, and it's exact rather than merely poetic.

Plain gradient descent is a ball with **no mass**. At every point it goes
precisely where the local slope says, which means in a narrow valley it
zig-zags — bouncing off one wall, then the other, making very little forward
progress along the valley itself.

Momentum gives the ball inertia. Now the consistent direction (along the valley)
accumulates step after step, while the oscillating directions (across the valley)
keep cancelling themselves out. The zig-zag averages away and the forward motion
adds up.

**Adam** adds a second idea on top: divide each parameter's step by a running
estimate of *that parameter's* gradient magnitude. Parameters with consistently
large gradients get proportionally smaller steps, and vice versa — so every
parameter ends up with its own effective learning rate, tuned automatically.

Which is exactly the problem we spotted in the very first plot, where `w` sprinted
and `b` crawled. Adam gives `b` a bigger step because `b`'s gradients are
consistently small.

That's why Adam works out of the box on so many problems, and why
`torch.optim.Adam(model.parameters(), lr=1e-3)` is the default line in most
training scripts you'll ever read.

Two exponential moving averages — one of the gradient, one of its square:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t
\qquad
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2
$$

Both start at zero, which biases them low for the first few steps, so they get
corrected:

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

And the update divides one by the root of the other:

$$
\theta_{t+1} = \theta_t - \eta\,\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

That division is the whole trick. $\hat{m}$ is "which way have I consistently been
going" (momentum) and $\sqrt{\hat{v}}$ is "how big are this parameter's gradients
typically" (the per-parameter scaling). Dividing one by the other gives a step
that's about the same *size* for every parameter regardless of its gradient
scale.

Defaults are $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, and you
should essentially never change them. The $\epsilon$ exists purely to avoid
dividing by zero and has no deeper meaning.

**AdamW** is Adam with one fix: weight decay applied directly to the parameters,
rather than folded into the gradient where the $\sqrt{\hat{v}}$ division was
silently rescaling it per-parameter. It's a small change, it's strictly better,
and it's what you should use.

## The same loop, for classification

Here's a nice surprise: essentially nothing structural changes.

Swap the loss for cross-entropy, put a
sigmoid on the output, and — through a small miracle we'll
come back to in a moment — the gradient comes out looking *identical*.

In [ ]:
passed = (score > 50).astype(int)

def fit_logistic(x, y, lr=0.5, steps=3000):
    X = np.column_stack([np.ones(len(x)), (x - x.mean()) / x.std()])  # standardised
    w = np.zeros(2)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-(X @ w)))       # sigmoid of the logits
        g = X.T @ (p - y) / len(y)           # <- the same shape as before
        w -= lr * g
    return w, X

w_log, Xs = fit_logistic(hours, passed)
p = 1 / (1 + np.exp(-(Xs @ w_log)))
print(f"training accuracy: {((p > 0.5).astype(int) == passed).mean():.1%}")

Look hard at that gradient line: `X.T @ (prediction - target) / n`.

That is *exactly* the linear-regression gradient, with a different `prediction`
plugged in. Different model, different loss, same expression.

It's not a coincidence and it's not a tidy-up. The sigmoid's derivative and
cross-entropy's derivative very nearly cancel each other out, leaving behind the
plain "prediction minus target." That elegance is one of the real reasons this
pairing became the standard — and softmax with cross-entropy does the identical
thing for many classes. You'll meet this cancellation again in chapter 9, where
it saves you a great deal of algebra.

And note the standardisation, quietly doing essential work in the first line.

Gradient descent on raw features whose scales differ by orders of magnitude is
miserable. The loss surface becomes a long, thin ravine, and no single learning
rate suits both directions — which you now recognise, because it's the `w`-versus-`b`
problem again, but much worse.

`(x - mean) / std` makes the bowl round. It's the most common preprocessing step
in the entire field and it's one line of broadcasting.

## The three ways it goes wrong

Almost every training failure you'll have is one of these three. Learn them now
and save yourself the evenings.

**Loss is `nan`.** Learning rate too high, or a `log(0)` somewhere. Clip
probabilities away from 0 and 1, then lower the rate by 10×.

**Loss doesn't move at all.** Learning rate too low, features not standardised,
or — the embarrassing one, and it happens to everybody — you aren't actually
updating the parameters. Print the gradient norm. If it's zero, your bug is
upstream of the optimiser and no amount of tuning will help.

**Loss falls and then rises.** The rate that was right at the start is too high
for the later, flatter part of training. Use a schedule that decays the rate over
time; cosine decay is the current default and it's one line in any framework.

**"Why do we subtract the gradient instead of adding it?"** The gradient points
*uphill* — toward increasing loss. We want to go down, so we go the other way.
The minus sign is the entire "descent" in gradient descent.

**"Where did `2 * (resid * x).mean()` come from?"** Straight out of the
derivation at the top of the page. `resid` is $r_i$, multiplying by `x` gives
$r_i x_i$, and `.mean()` does the $\frac{1}{n}\sum$. Read the code and the formula
side by side — they're the same sentence in two languages.

**"My loss went down but the parameters are still wrong."** Very likely you ran
out of steps rather than out of hill. Look at the `b` in the first cell: 200 steps
wasn't enough. Try 5000 and watch it arrive. A loss curve that's still visibly
descending at the end is telling you to keep going.

**"I don't understand why standardising matters so much."** Try it. The third
exercise below is exactly this experiment, and doing it once is worth more than
any explanation I can write — you'll watch a learning rate that worked perfectly
turn into `nan` purely because a feature got multiplied by 1000.

**"How would I ever guess the right learning rate on a real problem?"** You
wouldn't, and nobody does. You run the learning-rate finder above, or you try
powers of ten. This is genuinely empirical, and everybody — including the people
training frontier models — is doing the same slightly embarrassing thing.

In [ ]:
# 1. Add L2 regularisation to `fit`: penalise (lam * w**2), which adds
#    (2 * lam * w) to grad_w. Fit with lam = 0, 0.1, 1.0 and print w.
#    Which way does w move, and why?
#
# 2. Implement a learning-rate schedule: lr_t = lr0 / (1 + decay * t).
#    Does it reach a lower final loss than a constant rate?
#
# 3. Break it deliberately. Fit on `hours * 1000` without standardising.
#    Find the largest learning rate that does not produce nan.

print("replace me")

For 1, the penalty term goes *into the gradient* — it's an extra thing pushing
`w`, and you should be able to predict which direction before you run it.

For 3, start at `1e-4` and keep dividing by 100. The answer will be smaller than
you expect, and that's the lesson.

In [ ]:
def fit_reg(x, y, lr=0.01, steps=400, lam=0.0):
    w = b = 0.0
    for _ in range(steps):
        r = (w * x + b) - y
        w -= lr * (2 * (r * x).mean() + 2 * lam * w)
        b -= lr * 2 * r.mean()
    return w, b

for lam in [0.0, 0.1, 1.0, 10.0]:
    w_, b_ = fit_reg(hours, score, lam=lam)
    print(f"lam={lam:5.1f}   w={w_:6.3f}  b={b_:6.3f}")

The weight shrinks toward zero as the penalty grows. That's literally what the
word "shrinkage" means, and it's the L2 norm being used as a
regulariser. You've just built the thing chapter 6 is about, a day early.

Notice the bias is deliberately *not* penalised. It carries no complexity cost —
it just recentres the predictions, and shrinking it would only make the model
worse for no gain. Every library on Earth makes this same choice, usually without
mentioning it.

In [ ]:
def fit_sched(x, y, lr0=0.02, steps=300, decay=0.0):
    w = b = 0.0
    for t in range(steps):
        lr = lr0 / (1 + decay * t)
        r = (w * x + b) - y
        w -= lr * 2 * (r * x).mean()
        b -= lr * 2 * r.mean()
    return ((w * x + b - y) ** 2).mean()

print(f"constant lr : {fit_sched(hours, score, decay=0.0):.4f}")
print(f"decaying lr : {fit_sched(hours, score, decay=0.02):.4f}")

Decay wins, and the reason generalises beautifully: a large step is *right* early
on, when you're far from the bottom and want to cover ground. It's *wrong* late
on, when you're near the bottom and the large steps just bounce you around it
without settling.

Same walk, different terrain, different stride. Every learning-rate schedule ever
published is a variation on that one sentence.

In [ ]:
scaled = hours * 1000
for lr in [1e-4, 1e-6, 1e-8]:
    w_, b_ = fit_reg(scaled, score, lr=lr, steps=200)
    print(f"lr={lr:.0e}  ->  w={w_:.3e}  {'nan/inf' if not np.isfinite(w_) else 'ok'}")

Look at that. Multiplying one feature by 1000 divided the usable learning rate by
roughly a **million**.

The reason is worth knowing: the gradient scales with $x$, and the curvature of
squared error scales with $x^2$. So a thousand-fold change in the input becomes a
million-fold change in how far you're allowed to step.

This is why standardising isn't a nicety or a tidiness habit. Two features on
different scales turn your nice round bowl into a knife-edge ravine, and there is
no single learning rate that's right for both directions at once. One line of
preprocessing, and the problem simply stops existing.

Tomorrow, the question that decides whether any of this was worth doing: does it
work on data it has never seen?